# Big Five personality prediction on PANDORA ("PANDORA v3")

**Source:** Part 2 of the original master notebook (cells 19-98): sections S0-S11.

## 1. Objective and experimental scope
Predict the five Big Five trait scores (O, C, E, A, N; 0-100) of Reddit comment authors from text, evaluated as **binary** classification
(`score >= train median`) with **macro-F1** per trait, and compare:

```
                     ┌─ TF-IDF (150k) ─ SVD(300) ─┐
Text ──┬── BERT-MT ──┤                            ├── Ensemble (Ridge) ── predictions
       │             └─ 5 regression heads        │
       ├── BGE-M3 ── Ridge multi-output
       └── TF-IDF ── Ridge baseline / LogReg / NB / LinearSVM / LSTM / BiLSTM / GRU
```

Design choices stated in the original executive summary: continuous regression targets (`score/100`), one shared BERT with 5 heads, 500k BERT
training rows + 50k held-out rows for the ensemble, warm-up + frozen lower layers + weight-decay 0.1, thresholds optimised on validation, final score on test.
The original also stated the goal "F1 > 0.70"; **the only test score recorded in the notebook is a mean macro-F1 of 0.5745** (see §8), so that goal was not reached
in the recorded run.

> **Status:** not executed in the preparation environment (needs the PANDORA download ≈ 0.5 GB, ~50 GB RAM for the 1.9M×150k TF-IDF + SVD,
> and a GPU for BERT / BGE-M3; the original ran on a Colab A100 80 GB). Numbers in §8 are the ones recorded in the original outputs.
> The BERT, BGE-M3, ensemble-validation and comparison-table cells have **no recorded output** in the source.

In [ ]:
import sys
from pathlib import Path

# Locate the repository root (works from the repo root or from notebooks/) and make `src` importable.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").is_file())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import get_paths, get_device, set_seed, save_table, save_figure

paths = get_paths(create=True)      # data/, results/, cache/, checkpoints/ (override with ITC_* env vars)
SEED = 42
set_seed(SEED)                      # seed once, as the original notebook did
DEVICE = get_device()
print("repo root:", ROOT, "| device:", DEVICE)

## 2. Dataset loading and validation
`jingjietan/pandora-big5`: pre-defined train / validation / test splits (1,924,201 / 481,051 / 601,314 rows).

In [ ]:
from src.data import TRAITS, TRAIT_NAMES, load_pandora

df_train, df_val, df_test = load_pandora(cache_dir=paths.cache_dir)   # validates columns and score ranges
print(f"train {df_train.shape} | val {df_val.shape} | test {df_test.shape}")
df_train.head(3)

## 3. Preprocessing and exploratory checks
Cleaning (`clean_text_pandora`): lower-case, remove `@mentions`, HTML entities, newlines/tabs and non `[a-z0-9']` characters; rows whose cleaned text is empty are dropped
(original: 1,919,729 / 479,864 / 599,901 rows remain).

### Data-leakage considerations (kept from the original EDA)
1. Splits are the dataset's own. 8.27 % of training texts are duplicated; identical texts across splits would inflate generalisation estimates (this overlap was **not measured** in the notebook).
2. TF-IDF and SVD are fitted on the training split only.
3. Binarisation uses the **training** medians for all splits.
4. The loaded frame has no author identifier, so author-level overlap between splits cannot be checked - generalisation to unseen authors is not assessed.

In [ ]:
from src.preprocessing import clean_pandora_splits

df_train, df_val, df_test = clean_pandora_splits(df_train, df_val, df_test)
print("after cleaning:", df_train.shape, df_val.shape, df_test.shape)

desc = df_train[TRAITS].describe().round(2); display(desc)
print("duplicated texts (train):", int(df_train.duplicated(subset=["text"]).sum()))
save_table(desc.reset_index().rename(columns={"index": "statistic"}), paths.tables_dir, "personality", "trait_descriptives", split="train")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df_train[TRAITS].corr(), annot=True, cmap="vlag", vmin=-1, vmax=1, fmt=".2f", linewidths=.5, ax=ax)
ax.set_title("Correlation Matrix of OCEAN Traits (PANDORA Train)")
save_figure(fig, paths.figures_dir, "personality", "ocean_correlation", split="train"); plt.show()

## S3 - Labels: continuous + binary (train medians)
Regression targets = `score/100`; binary label = `score >= train median`; predictions are binarised with `median/100`.

In [ ]:
from src.preprocessing import binarize_traits, compute_train_medians, scale_traits

THRESHOLDS_MEDIAN = compute_train_medians(df_train, TRAITS)
print("Train medians:", THRESHOLDS_MEDIAN)
y_train_reg, y_val_reg, y_test_reg = (scale_traits(d, TRAITS) for d in (df_train, df_val, df_test))
y_train_bin, y_val_bin, y_test_bin = (binarize_traits(d, TRAITS, THRESHOLDS_MEDIAN) for d in (df_train, df_val, df_test))
print((y_train_bin.mean() * 100).round(2))

## S4 - TF-IDF baseline (Ridge regression, full training split, threshold = train median)

In [ ]:
import time
from src.features import PANDORA_TFIDF_CFG, fit_transform_tfidf
from src.models import make_ridge
from src.evaluation import evaluate_binary, find_best_threshold, mean_macro_f1_by_model

t0 = time.time()
vec, X_train_tfidf, (X_val_tfidf, X_test_tfidf) = fit_transform_tfidf(
    df_train["text_clean"], [df_val["text_clean"], df_test["text_clean"]], **PANDORA_TFIDF_CFG)
print(f"TF-IDF: {time.time()-t0:.1f}s | shapes {X_train_tfidf.shape} {X_val_tfidf.shape} {X_test_tfidf.shape}")

classical_results = []
for c in TRAITS:
    t0 = time.time()
    ridge = make_ridge(1.0, SEED).fit(X_train_tfidf, y_train_reg[c])
    t_fit = time.time() - t0
    pred_val = ridge.predict(X_val_tfidf)
    m = evaluate_binary(y_val_bin[c], (pred_val >= THRESHOLDS_MEDIAN[c] / 100.0).astype(int))
    m.update(dict(Trait=c, Trait_Name=TRAIT_NAMES[c], Model="Ridge (TF-IDF)", Time_s=round(t_fit, 1)))
    classical_results.append(m)
    print(f"{c} | Ridge TF-IDF | MacroF1 = {m['Macro_F1']:.4f} | {t_fit:.1f}s")
classical_df = pd.DataFrame(classical_results)
print("Mean Ridge TF-IDF:", round(classical_df.Macro_F1.mean(), 4))

### S4.3 - LogReg, Naive Bayes, Linear SVM (binary; trained on the **first 300,000** training rows; evaluated on the full validation split)

In [ ]:
from src.models import pandora_binary_models
from src.evaluation import text_classification_report

N_SAMPLES_ML = 300_000
X_train_ml, y_train_ml = X_train_tfidf[:N_SAMPLES_ML], y_train_bin[:N_SAMPLES_ML]
ml_results = []
for c in TRAITS:
    print(f"\n--- Trait: {c} ({TRAIT_NAMES[c]}) ---")
    for model_name, clf in pandora_binary_models(SEED).items():   # fresh model per trait
        t0 = time.time(); clf.fit(X_train_ml, y_train_ml[c]); t_fit = time.time() - t0
        pred_val = clf.predict(X_val_tfidf)
        m = evaluate_binary(y_val_bin[c], pred_val)
        m.update({"Trait": c, "Trait_Name": TRAIT_NAMES[c], "Model": model_name, "Time_s": round(t_fit, 1)})
        ml_results.append(m)
        print(f"[{model_name:19s}] Macro-F1: {m['Macro_F1']:.4f} | Acc: {m['Accuracy']:.4f} | {t_fit:.1f}s")
ml_df = pd.DataFrame(ml_results)
summary_ml = mean_macro_f1_by_model(ml_df); display(summary_ml)
classical_df = pd.concat([classical_df, ml_df], ignore_index=True)

from src.evaluation import plot_trait_comparison
fig = plot_trait_comparison(ml_df, "Comparaison des scores Macro F1 - Modèles ML Classiques sur TF-IDF")
save_figure(fig, paths.figures_dir, "personality", "classical_ml_macro_f1_by_trait", split="val"); plt.show()

*Refactoring note:* in the original, the `models` dict was created once and the same estimator objects were re-fitted for each trait; since each `fit` resets the model this is equivalent to
creating fresh estimators per trait as done above (same hyper-parameters, same seed).

### S4.5 - Recurrent models (multi-output, BCE loss; first 100,000 training rows / first 50,000 validation rows; 3 epochs)
Same right-padding / last-position caveat as in the emotion notebooks (`docs/methodology.md` §7). The per-trait score of the **last epoch** is reported (threshold 0.5).

In [ ]:
import torch, torch.nn as nn
from sklearn.metrics import f1_score
from src.preprocessing import build_vocab, encode_texts
from src.preprocessing.sequences import embedding_vocab_size
from src.models.rnn import PANDORA_RNN as P, RNNClassifier, make_loader, predict_sigmoid, train_fixed_epochs

texts_lstm = df_train["text_clean"].iloc[:P["n_train"]].tolist()
vocab = build_vocab(texts_lstm, P["vocab_size"])
y_tr = y_train_bin.iloc[:P["n_train"]].values
y_va = y_val_bin.iloc[:P["n_val"]].values
train_loader = make_loader(encode_texts(texts_lstm, vocab, P["max_len"]), y_tr, P["batch_size"], shuffle=True, y_dtype=torch.float32)
val_loader = make_loader(encode_texts(df_val["text_clean"].iloc[:P["n_val"]].tolist(), vocab, P["max_len"]), y_va, P["eval_batch_size"], y_dtype=torch.float32)

dl_results = []
for rnn_type in P["rnn_types"]:
    print(f"\n=== {rnn_type} ===")
    model = RNNClassifier(rnn_type, embedding_vocab_size(vocab), P["embed_dim"], P["hidden_dim"], num_outputs=5, dropout=P["dropout"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=P["lr"])
    last = {}
    def on_epoch_end(epoch, loss):
        probs = predict_sigmoid(model, val_loader, DEVICE)
        last["f1"] = [f1_score(y_va[:, i], (probs[:, i] > 0.5).astype(int), average="macro", zero_division=0) for i in range(5)]
        print(f"  Epoch {epoch+1}/{P['epochs']} | Loss: {loss:.4f} | mean macro-F1 (val): {np.mean(last['f1']):.4f}")
    train_fixed_epochs(model, train_loader, nn.BCEWithLogitsLoss(), optimizer, P["epochs"], DEVICE, on_epoch_end=on_epoch_end)
    for i, c in enumerate(TRAITS):
        dl_results.append({"Trait": c, "Trait_Name": TRAIT_NAMES[c], "Model": rnn_type, "Macro_F1": last["f1"][i]})
dl_df = pd.DataFrame(dl_results); display(mean_macro_f1_by_model(dl_df))
classical_df = pd.concat([classical_df, dl_df], ignore_index=True)
fig = plot_trait_comparison(dl_df, "Comparaison des architectures Deep Learning (LSTM vs BiLSTM vs GRU)", palette="coolwarm", ylim=(0.45, 0.62), figsize=(12, 5))
save_figure(fig, paths.figures_dir, "personality", "dl_macro_f1_by_trait", split="val"); plt.show()

## S5 - Multi-task DistilBERT (GPU)
Split inside the training set: **500k** rows train BERT; **50k** disjoint held-out rows will fit the ensemble; validation uses a fixed 20k subset for checkpoint selection.
Model: `distilbert-base-uncased`, CLS token → dropout 0.3 → 5 MLP heads (256, GELU), MSE on `score/100`, lowest 3 layers frozen, 2 epochs, batch 64, lr 2e-5,
weight-decay 0.1, warm-up 10 %, fp16, best checkpoint by mean macro-F1 at train-median thresholds. *Hardware:* the original recorded 7.66 min for this training on an A100.

In [ ]:
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, DataCollatorWithPadding, Trainer
from src.data import make_bert_ensemble_indices, sample_validation_indices
from src.models.hf_finetuning import (PANDORA_BERT as PB, MultiTraitBERT, build_training_arguments,
    make_multitrait_compute_metrics, multitrait_training_kwargs, predict_multitrait, trainer_tokenizer_kwargs)

if DEVICE != "cuda":
    raise RuntimeError("This section was run on a CUDA GPU. Skip to the next section or run on a GPU machine.")

idx_tr, idx_ens = make_bert_ensemble_indices(len(df_train), PB["n_train"], PB["n_ensemble"], SEED)
tokenizer_bert = AutoTokenizer.from_pretrained(PB["model_name"], cache_dir=str(paths.cache_dir))
tok = lambda batch: tokenizer_bert(batch["text"], truncation=True, max_length=PB["max_len"])

def make_ds(texts, labels):
    return HFDataset.from_dict({"text": texts, "labels": labels.tolist()}).map(tok, batched=True, remove_columns=["text"])

train_ds = make_ds(df_train["text_clean"].iloc[idx_tr].tolist(), y_train_reg.iloc[idx_tr].values.astype(np.float32))
val_sub = sample_validation_indices(len(df_val), PB["n_val_subset"], SEED)
val_ds = make_ds(df_val["text_clean"].iloc[val_sub].tolist(), y_val_reg.iloc[val_sub].values.astype(np.float32))

model_mt = MultiTraitBERT(PB["model_name"], PB["n_traits"], PB["dropout"], PB["freeze_layers"], cache_dir=paths.cache_dir).to(DEVICE)
print("trainable parameters:", f"{sum(p.numel() for p in model_mt.parameters() if p.requires_grad):,}")

args_bert = build_training_arguments(cpu=False, **multitrait_training_kwargs(str(paths.checkpoints_dir / "bert_mt"), cpu_mode=False, seed=SEED))
trainer = Trainer(model=model_mt, args=args_bert, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer_bert, return_tensors="pt"),
                  compute_metrics=make_multitrait_compute_metrics(THRESHOLDS_MEDIAN, TRAITS),
                  **trainer_tokenizer_kwargs(tokenizer_bert))
t0 = time.time(); trainer.train(resume_from_checkpoint=False); print(f"Total: {(time.time()-t0)/60:.2f} min")
m = trainer.evaluate(); print({k: round(v, 4) for k, v in m.items() if k.startswith("eval_f1_") or k == "eval_macro_f1_avg"})
trainer.save_model(str(paths.checkpoints_dir / "bert_mt_final")); tokenizer_bert.save_pretrained(str(paths.checkpoints_dir / "bert_mt_final"))

*Not reproduced:* the original also had a laptop-CPU fallback (1,000 samples, 20 steps, batch 2). It produced no recorded result and is omitted; the section above requires CUDA.

## S6 - BERT predictions on the held-out block, validation and test

In [ ]:
emb_dir = paths.cache_dir / "embeddings"; emb_dir.mkdir(exist_ok=True)
held_preds = predict_multitrait(trainer.model, tokenizer_bert, df_train["text_clean"].iloc[idx_ens].tolist(), DEVICE, PB["max_len"])
val_preds = predict_multitrait(trainer.model, tokenizer_bert, df_val["text_clean"].tolist(), DEVICE, PB["max_len"])
test_preds = predict_multitrait(trainer.model, tokenizer_bert, df_test["text_clean"].tolist(), DEVICE, PB["max_len"])
for name, arr in [("held", held_preds), ("val", val_preds), ("test", test_preds)]:
    np.save(emb_dir / f"bert_{name}_preds.npy", arr)
    print(f"{name:5s} | shape={arr.shape} min={arr.min():.3f} max={arr.max():.3f} mean={arr.mean():.3f}")

thr_arr = np.array([THRESHOLDS_MEDIAN[c] / 100.0 for c in TRAITS])
pred_bin_bert = (val_preds >= thr_arr).astype(int)
bert_df = pd.DataFrame([dict(evaluate_binary(y_val_bin.values[:, i], pred_bin_bert[:, i]), Trait=c, Trait_Name=TRAIT_NAMES[c], Model="DistilBERT MT")
                        for i, c in enumerate(TRAITS)])
print("Mean BERT MT (val):", round(bert_df.Macro_F1.mean(), 4))

## S7 - Ensemble: `[SVD(TF-IDF, 300) | BERT predictions]` → Ridge
Ridge per trait fitted on the 50k held-out block; threshold per trait chosen **on validation** (grid 0.20-0.81, macro-F1) and then applied unchanged to the test split.
The *validation* score of the ensemble is therefore optimistic (threshold tuned and scored on the same data); the *test* score is the honest one.

In [ ]:
from src.features import fit_truncated_svd
from src.models import fit_stacked_ridge_ensemble

svd, X_train_svd, (X_val_svd, X_test_svd) = fit_truncated_svd(X_train_tfidf, [X_val_tfidf, X_test_tfidf], n_components=300, seed=SEED)
print("SVD explained variance:", round(float(svd.explained_variance_ratio_.sum()), 3))

X_ens_tr = np.hstack([X_train_svd[idx_ens], held_preds]); y_ens_tr = y_train_reg.iloc[idx_ens].values.astype(np.float32)
X_ens_val = np.hstack([X_val_svd, val_preds]); X_ens_test = np.hstack([X_test_svd, test_preds])
ensemble_val_df, ensemble_test_df, BEST_THRESHOLDS = fit_stacked_ridge_ensemble(
    X_ens_tr, y_ens_tr, X_ens_val, y_val_bin.values, X_ens_test, y_test_bin.values, TRAITS, TRAIT_NAMES)
print(ensemble_test_df[["Trait", "Macro_F1", "Accuracy"]])
print("Mean ensemble macro-F1 | val:", round(ensemble_val_df.Macro_F1.mean(), 4), "| test:", round(ensemble_test_df.Macro_F1.mean(), 4))
save_table(ensemble_test_df, paths.tables_dir, "personality", "ensemble_trait_metrics", model="ensemble", split="test")

## S8 - BGE-M3 embeddings + multi-output Ridge (GPU; 300k training rows)
Embeddings (`BAAI/bge-m3`, max length 256, batch 128, normalised) of the first 300k BERT-training rows and of the whole validation split. Thresholds are tuned on validation
and scored on validation (optimistic, as above). No result is recorded in the source.

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from src.features import encode_with_sentence_transformer, load_sentence_encoder

bge = load_sentence_encoder("BAAI/bge-m3", device=DEVICE, cache_dir=paths.cache_dir, max_seq_length=256)
idx_bge = idx_tr[:300_000]
X_bge_tr = encode_with_sentence_transformer(bge, df_train["text_clean"].iloc[idx_bge].tolist())
X_bge_val = encode_with_sentence_transformer(bge, df_val["text_clean"].tolist())
ridge_bge = MultiOutputRegressor(make_ridge(1.0)).fit(X_bge_tr, y_train_reg.iloc[idx_bge].values.astype(np.float32))
pred_bge_val = ridge_bge.predict(X_bge_val)
rows = []
for i, c in enumerate(TRAITS):
    best_t, _ = find_best_threshold(y_val_bin[c].values, pred_bge_val[:, i])
    m = evaluate_binary(y_val_bin[c], (pred_bge_val[:, i] >= best_t).astype(int))
    m.update(dict(Trait=c, Trait_Name=TRAIT_NAMES[c], Model="BGE-M3+Ridge")); rows.append(m)
bge_df = pd.DataFrame(rows); print("Mean BGE-M3 (val):", round(bge_df.Macro_F1.mean(), 4))

## S10 - Final comparison (validation) and test score of the ensemble

In [ ]:
from src.utils import save_json_config
frames = [classical_df, bert_df, ensemble_val_df] + ([bge_df] if "bge_df" in globals() else [])
compare = pd.concat(frames, ignore_index=True)[["Trait", "Trait_Name", "Model", "Accuracy", "Macro_F1"]]
compare = compare.sort_values(["Trait", "Macro_F1"], ascending=[True, False])
save_table(compare, paths.tables_dir, "personality", "model_comparison", split="val")
display(compare.pivot_table(index="Model", values="Macro_F1", aggfunc="mean").round(4).sort_values("Macro_F1", ascending=False))

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=compare, x="Trait", y="Macro_F1", hue="Model", ax=ax)
ax.axhline(0.5, color="red", ls="--", lw=1, alpha=0.5, label="Hasard"); ax.set_ylim(0.4, 0.8)
ax.set_title("Comparaison finale v3 - Macro F1 (val)"); ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
save_figure(fig, paths.figures_dir, "personality", "comparison_v3", split="val"); plt.show()
print("FINAL TEST (ensemble) mean macro-F1:", round(ensemble_test_df.Macro_F1.mean(), 4))

save_json_config(dict(SEED=SEED, DEVICE=DEVICE, THRESHOLDS_MEDIAN=THRESHOLDS_MEDIAN, OPTIMIZED_THRESHOLDS=BEST_THRESHOLDS,
                      BERT=PB, TFIDF=PANDORA_TFIDF_CFG, SVD_DIM=300, ensemble="Ridge on [SVD(TF-IDF) | BERT preds]",
                      final_test_macro_f1=float(ensemble_test_df.Macro_F1.mean())),
                 paths.results_dir / "personality__config_v3.json")

*Removed from the original:* the line `ax.axhline(0.614, ..., label="v1 LogReg")` and the "gain" printout (`ensemble test − 0.614`). The value 0.614 is hard-coded in the original,
with no cell that produces it, so it cannot be traced to a result; a test-set score was also compared with a validation-set baseline there.

## S11 - Explainability (LIME), Extraversion
A small TF-IDF + logistic-regression pipeline (first 50k training rows, 20k features) explains one test text for the binary *Extraversion* label. Qualitative illustration only.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from src.evaluation.explain import explain_instance

xai_pipeline = make_pipeline(TfidfVectorizer(max_features=20000, ngram_range=(1, 2)), LogisticRegression(max_iter=1000, random_state=SEED))
xai_pipeline.fit(df_train["text_clean"].iloc[:50_000], y_train_bin["E"].iloc[:50_000])
text = df_test["text_clean"].iloc[100]
exp, weights, _ = explain_instance(xai_pipeline.predict_proba, text,
                                   ["Low Extraversion (Introverti)", "High Extraversion (Extraverti)"], num_features=10, top_labels=None)
for word, w in weights: print(f"  - {word:15s} : {w:+.4f}")
fig = exp.as_pyplot_figure(); fig.set_size_inches(10, 5); plt.tight_layout()
save_figure(fig, paths.figures_dir, "personality", "lime_explanation_extraversion", model="tfidf-logreg", split="test"); plt.show()

## Optional - cross-lingual embedding sanity check (original cell 97)
Cosine similarity of 2 EN/FR sentence pairs with BGE-M3 (GPU) or `all-MiniLM-L6-v2` (CPU fallback). Demonstration only; no quantitative claim.

In [ ]:
from sentence_transformers import util
from src.features import load_sentence_encoder

texts = ["I am really excited about this new deep learning project!",
         "Je suis vraiment très enthousiaste pour ce nouveau projet de deep learning !",
         "The results are quite disappointing and frustrating.",
         "Les résultats sont assez décevants et frustrants."]
enc = load_sentence_encoder("BAAI/bge-m3" if DEVICE == "cuda" else "sentence-transformers/all-MiniLM-L6-v2", device=DEVICE, cache_dir=paths.cache_dir, max_seq_length=None)
e = enc.encode(texts, convert_to_tensor=True)
print("same meaning EN/FR (excited):    ", round(util.cos_sim(e[0], e[1]).item(), 4))
print("same meaning EN/FR (disappointed):", round(util.cos_sim(e[2], e[3]).item(), 4))
print("different meaning:               ", round(util.cos_sim(e[0], e[3]).item(), 4))

## 8. Interpretation (only what was recorded in the original run)
Validation split, macro-F1 (`results/tables/personality__*__val__src-cell*.csv`):

| Model | O | C | E | A | N | Mean |
|---|---|---|---|---|---|---|
| Ridge on TF-IDF (full train) | 0.5851 | 0.5524 | 0.5477 | 0.5828 | 0.6054 | 0.5747 |
| Naive Bayes / LogReg / Linear SVM (300k rows), mean over traits | | | | | | 0.5892 / 0.5890 / 0.5801 |
| BiLSTM / LSTM / GRU (100k rows, 3 epochs), mean over traits | | | | | | 0.3693 / 0.3674 / 0.3652 |

Test split (`personality__ensemble_trait_f1__test__src-cell89.csv`), ensemble only: O 0.5681, C 0.5701, E 0.5588, A 0.5875, N 0.5878, **mean 0.5745**.

* All recorded scores are modest (0.55-0.61 macro-F1 on balanced binary targets, chance ≈ 0.5). The recorded ensemble test mean (0.5745) is essentially equal to the TF-IDF Ridge validation mean (0.5747),
  but these are different splits and different threshold rules, so no gain from BERT/SVD stacking can be claimed from the source.
* The three recurrent models score *below* 0.5 macro-F1 (≈ 0.37) after 3 epochs with training loss ≈ 0.69, i.e. they did not learn a useful signal under this configuration
  (the last-position/right-padding issue is a candidate cause, untested).
* Neuroticism and Openness are the traits with the highest Ridge scores; Extraversion the lowest. Differences between traits are ~0.05 and come from one run.
* **Not available in the source:** BERT-MT metrics (only the 7.66 min training time), BGE-M3 metrics, the validation score of the ensemble, and the comparison table (cells 66, 69, 72, 75, 88 have no output).
* The stated target (F1 > 0.70) is not supported by any recorded result.